In [ ]:
# @colab-setup
# Run this cell first. On Colab it installs the deps; locally it is a no-op.
# cioos-metadata-conversion is not on PyPI — install from GitHub.
import sys
if "google.colab" in sys.modules:
    %pip install -q requests
    %pip install -q git+https://github.com/cioos-siooc/cioos-metadata-conversion.git


# 04 — OBIS UUID → CIOOS metadata record

**Time budget:** ~10 min · **Goal:** show the metadata side of the pipeline. Not just *fetching* OBIS records, but *mapping* them into CIOOS schema and deriving Essential Ocean Variables.

All of the heavy lifting is in `cioos_metadata_conversion/load_from/obis.py`. We'll call it directly here, then run the CLI to render an ISO 19115-3 XML.

Two independent EOV signals get unioned:

1. **Taxonomy** (`/v3/facet?facets=class`) — 237 class names → EOV codes, with gating thresholds (e.g. ≥ 5 % rule for cover EOVs, plankton-net heuristics).
2. **Measurements** (`/v3/occurrence?mof=true`) — eMoF measurement types → EOV codes, via a P01-code lookup with a free-text fallback.

Then the merge logic drops the `other` fallback if real measurements are present (`load_from/obis.py:1114–1130`).

In [ ]:
from cioos_metadata_conversion.load_from.obis import (
    retrieve_obis_metadata,
    fetch_eovs_from_taxonomy,
    fetch_eovs_from_measurements,
    map_obis_to_cioos,
    _map_measurement_pair,
    TAXON_CLASS_TO_EOV,
)
import requests
import json

DEMO_UUID = 'd895e645-a98d-4720-b6fb-332929190f36'  # Maritimes Spring RV Surveys

## 1. End-to-end — `retrieve_obis_metadata`

One call. Hits OBIS three times under the hood (dataset, facet, occurrence/mof) and returns a CIOOS-shaped dict.

In [ ]:
cioos_record = retrieve_obis_metadata(DEMO_UUID)
print('Top-level CIOOS keys:')
print(' ', sorted(cioos_record.keys()))

In [ ]:
for k in ('title', 'datasetIdentifier', 'license', 'keywords', 'eov', 'extent'):
    v = cioos_record.get(k)
    if isinstance(v, (dict, list)):
        print(f'{k}:')
        print(json.dumps(v, indent=2, default=str)[:600])
    else:
        print(f'{k}: {str(v)[:200]}')
    print()

## 2. Taxonomy → EOV path

Same `/v3/facet` call we made by hand in Notebook 2, but with a CIOOS-aware mapper on top.

In [ ]:
# Raw signal: taxonomic class facets
facets = requests.get(
    'https://api.obis.org/v3/facet',
    params={'datasetid': DEMO_UUID, 'facets': 'class'},
    timeout=30,
).json()['results']['class']

for f in facets[:8]:
    cls = f['key']
    eov = TAXON_CLASS_TO_EOV.get(cls, '— unmapped —')
    print(f"  {cls:<25} {f['records']:>8,}  →  {eov}")

In [ ]:
tax_eovs = fetch_eovs_from_taxonomy(DEMO_UUID)
print('Taxonomy-derived EOVs:', tax_eovs)

## 3. Measurement → EOV path

OBIS's eMoF (extended Measurement or Fact) records carry CTD-style observations alongside the biological occurrences. The mapper extracts a sample, looks for [BODC P01](https://vocab.nerc.ac.uk/collection/P01/current/) URIs first, then falls back to free-text matching on `measurementType`.

In [ ]:
# Demonstrate the underlying mapper on a couple of pairs
examples = [
    ('Sea water temperature', 'http://vocab.nerc.ac.uk/collection/P01/current/TEMPPR01/'),
    ('Sea surface temperature', None),
    ('Salinity',                None),
    ('Air temperature',         None),  # explicitly suppressed
    ('pH',                      None),  # not strong inorganic carbon on its own
]
for m_type, m_id in examples:
    print(f'  {m_type!r:<35} {str(m_id)[-30:]:<32} → {_map_measurement_pair(m_type, m_id)}')

In [ ]:
# Real call — pulls a sample of eMoF rows from the dataset and maps them.
mof_eovs = fetch_eovs_from_measurements(DEMO_UUID)
print('Measurement-derived EOVs:', mof_eovs)
print()
print('Empty list? Check the loguru DEBUG output above for an "unmapped eMoF')
print('measurements" line — for the Maritimes Spring RV dataset, e.g.:')
print('  sea_surface_temperature → P07/CFSN0381 (CF Standard Name URI)')
print('  Sea_Bottom_Temperature  → P07/CFSN0335 (CF Standard Name URI)')
print('The mapper currently only handles BODC P01 codes (and free-text).')
print('Real signal is in the dataset; the mapper just does not know how to')
print('translate P07 / CF Standard Name URIs yet — a known gap.')

## 4. Merge

The final EOV list on the CIOOS record is the **union** of taxonomy + measurement signals — and the `other` fallback gets dropped if there was any real measurement signal.

In [ ]:
print('Taxonomy:    ', sorted(set(tax_eovs)))
print('Measurements:', sorted(set(mof_eovs)))
print('Final (CIOOS):', sorted(set(cioos_record.get('eov', []))))

## 5. Render to ISO 19115-3 XML

Same record, now via the CLI shipped with `cioos-metadata-conversion`. The `--input-schema obis` flag tells it the input is an OBIS UUID, not a JSON file (`__main__.py:36–68`, `record.py:104,203`).

In [ ]:
import subprocess, tempfile, os

with tempfile.TemporaryDirectory() as tmp:
    out_file = os.path.join(tmp, f'{DEMO_UUID}.xml')
    cmd = [
        'cioos_metadata_conversion', 'convert',
        '--input',         DEMO_UUID,
        '--input-schema',  'obis',
        '--output-format', 'iso19115-3_xml',
        '--output-file',   out_file,
    ]
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print('returncode:', result.returncode)
    if result.stderr:
        print('stderr (head):')
        print('\n'.join(result.stderr.splitlines()[:20]))
    if os.path.exists(out_file):
        with open(out_file) as f:
            head = ''.join(f.readlines()[:25])
        print('\n--- XML head ---')
        print(head)

## Wrap-up

We've now seen the same OBIS dataset four ways:

1. **pyobis** — three lines, exploratory.
2. **REST** — full control, faceting, cursor pagination.
3. **Parquet + DuckDB** — bulk + analytical, the explore-cioos production path.
4. **`load_from.obis`** — schema mapping + EOV derivation, the metadata-catalogue path.

Pick by job, not by habit:

- One-off look at a species or region → pyobis.
- Custom filters / new endpoint → REST.
- Anything analytical or bulk → Parquet + DuckDB.
- Pulling an OBIS dataset *into* CKAN/ISO catalogue → `load_from.obis`.